# FLATUP GYM ポーズ画像ジェネレーター(Google Colab用)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flatup1/openqlow/blob/main/girl-power-op/colab_generate_poses.ipynb)

`chara.png`(キャラの基準画像)を参照しながら、アニメに必要なポーズ画像を
**無料のColab GPU** で生成するノートブックです。すべてブラウザだけで完結します。

## 使い方(かんたん3ステップ)
1. 上のメニュー **ランタイム → すべてのセルを実行** をクリック(コードの編集は不要)
2. セル③で **`chara.png`** をアップロード(それ以外は自動で進みます)
3. 最後に **`poses.zip`** がダウンロードされる → 中身を `girl-power-op/poses/` に入れる

生成される9ポーズ(idle / punch1 / punch2 / jump / win / nervous / fall / kick / wave)は
このページ内にその場で表示されます。気に入らなければ `SEED` の数字を変えて作り直せます。

## ⚠️ クラッシュを防ぐコツ
- **セルは上から順に1回ずつ**。同じセルを何度も押さない
- やり直したいときは必ず **ランタイム → セッションを再起動** してから **すべてのセルを実行**
- このノートブックはメモリ節約の工夫(モデル二重読み込み防止・1枚ごとの掃除)を入れてあります

In [ ]:
# ① GPUが使えるか確認(「Tesla T4」などと出ればOK)
!nvidia-smi -L

In [ ]:
# ② 必要な部品を入れる(1〜2分)
!pip -q install diffusers transformers accelerate safetensors

In [ ]:
# ③ キャラの基準画像(chara.png)をアップロード
from google.colab import files
from PIL import Image
import io

up = files.upload()  # ファイル選択ダイアログが出ます
ref_image = Image.open(io.BytesIO(list(up.values())[0])).convert('RGB')
display(ref_image.resize((256, int(256 * ref_image.height / ref_image.width))))
print('この画像を参照してポーズを作ります')

In [ ]:
# ④ 画像生成AI(SDXL)+ 参照画像アダプター(IP-Adapter)を読み込む(数分)
#    ※ このセルは1回だけ実行。2回目以降は自動でスキップ(メモリ節約=クラッシュ防止)
import torch, gc
from diffusers import StableDiffusionXLPipeline

if 'pipe' not in globals():
    pipe = StableDiffusionXLPipeline.from_pretrained(
        'stabilityai/stable-diffusion-xl-base-1.0',
        torch_dtype=torch.float16, variant='fp16', use_safetensors=True,
    )
    # IP-Adapter: 「この画像と同じキャラで」と伝えるための部品
    pipe.load_ip_adapter('h94/IP-Adapter', subfolder='sdxl_models',
                         weight_name='ip-adapter_sdxl.bin')
    pipe.enable_model_cpu_offload()  # 無料GPUのメモリでも動くように
    pipe.enable_vae_slicing()        # メモリ節約(クラッシュ防止)
    pipe.enable_attention_slicing()  # メモリ節約(クラッシュ防止)
    print('準備OK(モデルを読み込みました)')
else:
    print('準備OK(読み込み済みなのでスキップしました)')

pipe.set_ip_adapter_scale(0.7)   # 参照画像にどれだけ似せるか(0.5〜0.9で調整)

In [ ]:
# ⑤ ポーズを生成する(1枚あたり1〜2分。全部で10〜20分ほど)
import os, gc
os.makedirs('poses', exist_ok=True)

SEED = 42  # 数字を変えると別の結果になる(気に入るまで変えてOK)

# 全ポーズ共通のスタイル指定(FLATUP GYMの共通ルール)
STYLE = ('warm 3D animated toy-like movie style, soft cinematic lighting, '
         'friendly expression, cute original chibi gym mascot, full body, '
         'vertical composition, character centered, feet at the bottom, '
         'consistent outfit and hairstyle and colors, clean bright gym background')
NEGATIVE = ('scary, aggressive, angry, violent, injury, text, watermark, logo, '
            'extra limbs, deformed hands, lowres, blurry')

# アニメで使う全ポーズ(貼り付け不要。いらないポーズは行頭に # を付けて消せる)
POSES = {
    # --- 基本(にぎやか版アニメで使う) ---
    'idle':    'standing in a relaxed kickboxing guard pose',
    'punch1':  'throwing a small left punch, dynamic but friendly',
    'punch2':  'throwing a small right punch, energetic and joyful',
    'jump':    'jumping happily in the air, arms up',
    'win':     'cheerful victory pose, smiling with confidence',
    # --- 感情(FLATUP GYM の物語版アニメで使う) ---
    'nervous': 'standing nervously, holding oversized boxing gloves close to the chest, uncertain shy eyes, feet turned inward',
    'fall':    'sitting on the gym mat after a harmless failed punch attempt, embarrassed but charming, looking down',
    'kick':    'performing a small friendly high kick, energetic and joyful',
    'wave':    'waving hello with one glove, big friendly smile',
}

for name, action in POSES.items():
    g = torch.Generator('cuda').manual_seed(SEED)
    img = pipe(
        prompt=f'{action}, {STYLE}',
        negative_prompt=NEGATIVE,
        ip_adapter_image=ref_image,
        num_inference_steps=30,
        width=768, height=1152,   # 縦長(メモリ安全側にすこし小さめ)
        generator=g,
    ).images[0]
    img.save(f'poses/{name}.png')
    print(f'--- {name}.png ---')
    display(img.resize((192, 288)))  # その場でブラウザ確認
    gc.collect(); torch.cuda.empty_cache()  # 1枚ごとにメモリを掃除(クラッシュ防止)

print('ぜんぶ完成! つぎのセル⑥でダウンロードしてください')

In [ ]:
# ⑥ まとめてダウンロード
!zip -q -r poses.zip poses
from google.colab import files
files.download('poses.zip')

## つぎにやること

1. `poses.zip` を解凍して、中の画像を `girl-power-op/poses/` に入れる
2. `npm run video` → キャラがポーズを切りかえる本物のアニメになる
3. ブラウザで確認したいだけなら `index.html?loop=1` を開けばループ再生される

## うまくいかないとき

- **キャラが似ていない** → ④の `set_ip_adapter_scale` を `0.8〜0.9` に上げる
- **ポーズが変わらない** → 逆に `0.5〜0.6` に下げる(似せる力を弱めるとポーズが自由になる)
- **手や足が変** → ⑤の `SEED` を変えて引き直す(ガチャと同じ)
- **メモリ不足エラー** → ランタイム → セッションを再起動 して②から実行し直す